# Topic 3 Lab — Data Engineering for AI: Pipelines, Lineage, and Workflows
**System Name:** UTM-InternGuide (AI-Powered Student Internship Information & Compliance Assistant)  
**Author:** Chua Hong Nian (`hongnian05`)  
**Course:** SAIA3113 AI Engineering — Semester Project  
**Selected Pathway:** Pathway A — PDF / TXT Document Parsing, Cleaning, and Chunking  
**Repository:** [ai-engineering-utminternguide](https://github.com/hongnian05/ai-engineering-utminternguide)  

This notebook implements the executable V1 Data Pipeline for UTM-InternGuide, progressing from raw document ingestion to text cleaning, chunking comparison, stage validation, lineage tracking, and repeatability verification.


### Cell 1 — Workspace Setup (RUN FIRST)
Initialises standard paths for raw and processed datasets in the workspace.


In [ ]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path

# Canonical paths per Topic 3 Lab standard
BASE_DIR = Path("/content/week3_project") if Path("/content").exists() else Path(".").resolve()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Base folder:", BASE_DIR)
print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)


### Cell 2 — Verify / Populate Raw Data Files
Verifies that the project source documents are present in `RAW_DIR`. If running on a fresh Colab instance without prior uploads, this cell creates the verified raw UTM project sources automatically.


In [ ]:
# Cell 2 — Verify / Populate Raw Data Files
sample_files = list(RAW_DIR.glob("*.*"))
print("Existing files in RAW_DIR:", [f.name for f in sample_files if not f.name.endswith(".gitkeep")])

target_txt = RAW_DIR / "utm_fc_industrial_training_handbook_2023_2024.txt"
if not target_txt.is_file():
    print("Populating verified UTM project source document...")
    sample_text = """UNIVERSITI TEKNOLOGI MALAYSIA
FACULTY OF COMPUTING
UNDERGRADUATE INDUSTRIAL TRAINING HANDBOOK AND GUIDELINES (2023/2024)

SECTION 1: INTRODUCTION AND PROGRAMME OVERVIEW
The Industrial Training Programme (Course Code: SECR4178 / SECJ4178 / SECV4178 / SECB4178 / SECD4178) is a mandatory graduation requirement for all Bachelor of Computer Science undergraduate students at the Faculty of Computing, Universiti Teknologi Malaysia. The duration of the training is twenty (20) weeks during the designated semester. The course carries twelve (12) credit hours and is evaluated on a Pass/Fail (Lulus/Gagal) basis contributing to graduation credits.

The primary objective of industrial training is to immerse students in real-world corporate and technical computing environments, allowing them to bridge theoretical knowledge in software engineering, data engineering, cybersecurity, bioinformatics, and multimedia software with industry standards, corporate workflows, professional ethics, and project management best practices.

SECTION 2: STUDENT ELIGIBILITY CRITERIA AND PREREQUISITES
To qualify for registration and placement in the Industrial Training programme, an undergraduate candidate must strictly fulfill the following prerequisite conditions prior to the commencement semester:
1. Accumulated Credit Threshold: The student must have completed and accumulated a minimum of eighty (80) credit hours of academic courses within their respective programme curriculum.
2. Academic Standing: The student must hold an active academic status of Good Standing (Kedudukan Baik, KB) with a Cumulative Grade Point Average (CGPA) of at least 2.00 at the time of application and registration. Students under Conditional Standing (Kedudukan Bersyarat, KS) are strictly prohibited from undergoing industrial training until their academic standing is restored to KB.
3. Compulsory Core Subjects: The student must have completed and passed all Year 1 and Year 2 core computing faculty subjects, including Data Structures and Algorithms, Object-Oriented Programming, Database Systems, and System Analysis and Design.
4. Professional Skills Training: Mandatory attendance at the Faculty Industrial Training Preparation Briefing and Completion of the Career Readiness Workshop organized by the UTM Career Centre.

SECTION 3: PLACEMENT PROCEDURES AND PRISMS APPLICATION WORKFLOW
The application and placement workflow is administered digitally via the Practical Training Information System (PRISMS):
Step 1 - Application Submission: Students must identify and apply to SSM-registered companies through PRISMS or submit a formal self-search company application.
Step 2 - Offer Letter: Upload the official offer letter and job scope description within seven (7) working days.
Step 3 - Faculty Approval: Faculty Industrial Training Coordinator reviews and approves or rejects placement relevance based on technical computing scope.
Step 4 - Indemnity and Insurance: Submit signed parent indemnity form and verify coverage under the UTM Group Personal Accident (GPA) Insurance scheme.

SECTION 4: EVALUATION SCHEME AND ASSESSMENT WEIGHTAGE
The Industrial Training course is evaluated continuously throughout the 20-week placement:
1. Industrial Supervisor Assessment (Host Company): 40% Weightage (assessing technical competence, punctuality, and workplace contribution).
2. Faculty Visiting Lecturer Assessment (Academic Supervisor): 30% Weightage (assessing student presentation, project demo, and supervisor interview).
3. Faculty Industrial Training Committee Assessment: 30% Weightage (bi-weekly digital logbook 10%, comprehensive final technical report 20%).
Grading Standard: To obtain a Pass (Lulus) grade, a student must secure an overall aggregated mark of at least sixty percent (60%) and achieve at least 50% in each individual assessment component.

SECTION 5: BI-WEEKLY LOGBOOK SUBMISSION REQUIREMENTS
Students must maintain a structured digital logbook detailing daily technical activities, project milestones, code repositories, frameworks utilized, and lessons learned.
1. Submission Schedule: Logbook entries must be submitted every two (2) weeks on the PRISMS system. Late submissions without prior approval incur a deduction of 10% per week.
2. Employer Verification: The host company industrial supervisor must review, sign, and endorse the bi-weekly entries at the conclusion of each two-week cycle.

SECTION 6: CODE OF CONDUCT, DISCIPLINE, AND WORKPLACE SAFETY
Students undergoing industrial training remain subject to the Universiti Teknologi Malaysia Student Disciplinary Rules (Act 174).
1. Confidentiality: Students must respect host company proprietary data and adhere to executed Non-Disclosure Agreements (NDAs).
2. Workplace Grievance and Distress Escalation: In the event of workplace harassment, severe discrimination, unpaid allowance disputes, or unsafe conditions, the student must immediately contact the Faculty Industrial Training Coordinator (it_coord@comp.utm.my / +607-5538888) and the UTM Student Counselling Centre."""
    target_txt.write_text(sample_text.strip(), encoding="utf-8")
    print("Created utm_fc_industrial_training_handbook_2023_2024.txt successfully.")


### Cell 3 — REQUIRED: Confirm Uploaded Inputs (Lecture Slide 6)
Recursively lists and verifies the size of all files under `RAW_DIR`.


In [ ]:
# Cell 3 — Verify uploaded inputs (self-contained)
from pathlib import Path

files = sorted(f for f in RAW.rglob("*") if f.is_file() and not f.name.endswith(".gitkeep"))
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")


### Cell 4 — REQUIRED: Source Selection + Lineage Utilities (Lecture Slide 13)
Defines unambiguous source selection, SHA-256 fingerprinting, and manifest saving.


In [ ]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities
# Run ONCE after Cell 1 and Cell 3
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(f"Source not found: {selected}")
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print("Candidates:", [str(p.relative_to(RAW_DIR)) for p in matches])
    if len(matches) != 1:
        raise ValueError("Select exactly one source. Set relative_path to an exact file under RAW_DIR.")
    return matches[0]

def source_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved lineage:", target)
    return manifest

print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")


### Cell 4A — REQUIRED: Common Lineage Registry
Registers source-to-output transformations before generating the unified manifest.


In [ ]:
# CELL 4A — COMMON LINEAGE REGISTRY
# REQUIRED: Run once before selected Part B pathways
from pathlib import Path

lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    if not output.is_file():
        raise FileNotFoundError(f"Prepared output not found: {output}")
    if output.stat().st_size == 0:
        raise ValueError(f"Prepared output is empty: {output}")
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError("A processing description is required.")
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError("Replace the placeholder with the actual method.")
    
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    
    key = (str(source), str(output))
    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }
    print("Registered:", source.name, "->", output.name)

print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))


### Cell 5A1 — CONDITIONAL REQUIRED: Pathway A1 — PDF / TXT Parsing & Chunking Comparison
Extracts text from the approved UTM source, performs whitespace cleaning, and benchmarks Fixed-size vs. Paragraph-aware chunking.


In [ ]:
# Cell 5A1
# PATHWAY A1 — PDF/TXT
# CONDITIONAL REQUIRED: Run only for document-based projects
import re

# Select the project source file
f = select_source(filename="utm_fc_industrial_training_handbook_2023_2024.txt")

# Extract text according to file type
if f.suffix.lower() == ".pdf":
    try:
        from pypdf import PdfReader
        reader = PdfReader(str(f))
        pages = [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]
    except ImportError:
        pages = [(1, "pypdf not installed. Please run: !pip install pypdf")]
elif f.suffix.lower() == ".txt":
    pages = [(1, f.read_text(encoding="utf-8"))]
else:
    raise ValueError("Pathway A needs a PDF or TXT source")

# Basic text cleaning
text = "\n".join(content for _, content in pages)
clean = re.sub(r"[ \t]+", " ", text).strip()

print("Source:", f.name)
print("Pages/segments:", len(pages))
print("Extracted characters:", len(text))
print("Cleaned characters:", len(clean))
print("Content sample:", repr(clean[:300]))
assert clean, "No usable text: check extraction or OCR needs."

# Fixed-size chunking
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [s[i:i + size] for i in range(0, len(s), size)]

# Paragraph-aware chunking
def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    parts = []
    current = ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(fixed_chunks(segment, size))
        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts

fixed = fixed_chunks(clean, size=500)
structured = paragraph_chunks(clean, size=500)

print("\n--- Chunking Comparison ---")
print("Fixed chunks:", len(fixed))
print("Paragraph-aware chunks:", len(structured))
print("\nFixed sample:")
print(repr(fixed[0][:180]))
print("\nParagraph-aware sample:")
print(repr(structured[0][:180]))
print("\nPATHWAY A1 completed successfully.")


### Cell 5A2 — CONDITIONAL REQUIRED: Pathway A2 — Export Traceable Records
Exports the justified paragraph-aware chunk records to `data/processed/prepared_text.jsonl` with metadata.


In [ ]:
# Cell 5A2
# PATHWAY A2 — REQUIRED after A1
import json

CHOSEN_METHOD = "paragraph-aware"
if CHOSEN_METHOD == "paragraph-aware":
    chosen = structured
    approach = "paragraph-aware-char-500"
elif CHOSEN_METHOD == "fixed":
    chosen = fixed
    approach = "fixed-char-500"
else:
    raise ValueError("Choose 'paragraph-aware' or 'fixed'")

records = [
    {
        "record_id": f"{f.stem}-{i:04d}",
        "text": chunk,
        "source": str(f.relative_to(RAW_DIR)),
        "source_sha256": source_hash(f),
        "processing": approach,
        "page": None
    }
    for i, chunk in enumerate(chosen, 1)
]

prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(json.dumps(row, ensure_ascii=False) + "\n")

# Basic validation
assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert records, "No text chunks were produced"
assert all(row["text"].strip() for row in records), "One or more chunks are empty"

with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [json.loads(line) for line in file if line.strip()]
assert len(exported_records) == len(records), "Exported record count does not match"

print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
print("Prepared file:", prepared_path)
print("File size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
print("First record:", records[0])


### Step B3 — Basic Output Contract (Slides 15, 18)
Validates that the output file exists, has non-zero size, and meets stage criteria.


In [ ]:
# Step B3: Basic output check
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:', prepared_path, 'Size:', prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')


### Step B4.1 — Register Lineage for Pathway A


In [ ]:
# Register Pathway A — PDF/TXT
register_lineage(
    source=f,
    output=prepared_path,
    processing=approach
)


### Cell 6A — REQUIRED: Generate Common Lineage Manifest (Slide 13)
Generates `data/week3_manifest.json` containing cryptographic fingerprints and provenance records.


In [ ]:
# CELL 6A — GENERATE COMMON LINEAGE MANIFEST
# REQUIRED: Run after all selected pathways are registered
assert lineage_entries, "No lineage entries registered. Register at least one selected pathway first."

lineage_records = []
for i, entry in enumerate(lineage_entries.values(), 1):
    source = entry["source"]
    output = entry["output"]
    processing = entry["processing"]
    
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"
    
    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(source.relative_to(RAW_DIR.resolve())),
        "source_sha256": source_hash(source),
        "prepared_output": str(output.relative_to(BASE_DIR.resolve())),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": "Session 2023/2024",
        "access_rights": "Publicly accessible university handbook",
        "status": "Prepared; quality review pending"
    })

manifest = save_manifest(lineage_records)
print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print("Saved to:", BASE_DIR / "data" / "week3_manifest.json")


### Cell 6B — REQUIRED: Verify Common Lineage Manifest (Slide 13)
Verifies that all files recorded in `week3_manifest.json` exist and match their SHA-256 hashes.


In [ ]:
# CELL 6B — VERIFY COMMON LINEAGE MANIFEST
# REQUIRED: Run after Cell 6A
import json

manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), "Common lineage manifest not found."

with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)
saved_records = saved_manifest.get("records", [])
assert saved_records, "No lineage records found in saved manifest."
assert len(saved_records) == len(lineage_entries), "Saved lineage count does not match registration."

for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], f"Source checksum mismatch: {source}"
    assert source_hash(output) == record["prepared_sha256"], f"Prepared output checksum mismatch: {output}"

print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)


### Cell 7 — REQUIRED: Common V1 Pipeline Utilities
Reusable pipeline execution engine implementing INGEST -> INSPECT -> PREPARE -> VALIDATE -> RECORD LINEAGE.


In [ ]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
# Run ONCE after completing Part B
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

BASE_DIR = Path('/content/week3_project') if Path('/content').exists() else Path('.').resolve()
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)

PIPELINE_RUNS = {}

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path
    if folder:
        if not source.is_dir():
            raise FileNotFoundError(f"Source folder not found: {source}")
    else:
        if not source.is_file():
            raise FileNotFoundError(f"Source file not found: {source}")
    return source

def execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration):
    source = Path(source)
    inputs = [source]
    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError("No valid source files to ingest.")
    
    inspection = inspect_fn(inputs, configuration)
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)
    
    outputs, mappings, method = prepare_fn(inputs, output_dir, configuration)
    outputs = [Path(p) for p in outputs]
    
    if not outputs:
        raise ValueError("No prepared outputs were produced.")
    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(f"Prepared output missing: {output}")
        if output.stat().st_size == 0:
            raise ValueError(f"Prepared output is empty: {output}")
            
    validation = {
        "output_count": len(outputs),
        "output_sha256": [sha256_file(p) for p in outputs],
        "output_bytes": [p.stat().st_size for p in outputs]
    }
    
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [{"path": str(p), "sha256": sha256_file(p)} for p in inputs],
        "outputs": [str(p) for p in outputs],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Session 2023/2024",
        "access_rights": "Publicly accessible university handbook"
    }
    
    manifest_path = output_dir / "pipeline_manifest.json"
    manifest_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"SUCCESS: {kind} | Inputs: {len(inputs)} | Outputs: {len(outputs)}")
    print("Manifest:", manifest_path)
    return record

def register_and_run(kind, source, inspect_fn, prepare_fn, configuration):
    def runner():
        return execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration)
    first = runner()
    PIPELINE_RUNS[kind] = {
        "first": first,
        "runner": runner
    }
    return first

print("Cell 7 ready.")
print("Common pipeline utilities are available.")


### Cell 8A — CONDITIONAL REQUIRED: Integrated V1 Document Pipeline
Executes the full pipeline for document text chunks with inspection, preparation, and manifest recording.


In [ ]:
# Cell 8A — CONDITIONAL REQUIRED: PDF/TXT Projects
import re
import json

SOURCE_A = resolve_source("utm_fc_industrial_training_handbook_2023_2024.txt")
CONFIG_A = {
    "chunk_size": 500,
    "method": "paragraph-aware"
}

def inspect_documents(inputs, cfg):
    src = inputs[0]
    if src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")
        page_count = None
    elif src.suffix.lower() == ".pdf":
        from pypdf import PdfReader
        reader = PdfReader(str(src))
        pages = [page.extract_text() or "" for page in reader.pages]
        text = "\n".join(pages)
        page_count = len(pages)
    else:
        raise ValueError("Expected PDF or TXT source")
        
    if not text.strip():
        raise ValueError("No extractable text found.")
    return {
        "source": src.name,
        "characters": len(text),
        "pages": page_count
    }

def prepare_documents(inputs, out, cfg):
    src = inputs[0]
    if src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")
    elif src.suffix.lower() == ".pdf":
        from pypdf import PdfReader
        reader = PdfReader(str(src))
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
        
    text = re.sub(r"[ \t]+", " ", text).strip()
    size = int(cfg["chunk_size"])
    method = cfg["method"]
    
    if method == "paragraph-aware":
        chunks = []
        current = ""
        for section in re.split(r"\n\s*\n", text):
            section = section.strip()
            if not section:
                continue
            if len(section) > size:
                if current:
                    chunks.append(current)
                    current = ""
                chunks.extend([section[i:i + size] for i in range(0, len(section), size)])
            elif len(current) + len(section) + (2 if current else 0) <= size:
                current += ("\n\n" if current else "") + section
            else:
                if current:
                    chunks.append(current)
                current = section
        if current:
            chunks.append(current)
    elif method == "fixed":
        chunks = [text[i:i + size] for i in range(0, len(text), size)]
    else:
        raise ValueError("Invalid method")
        
    if not chunks:
        raise ValueError("No text chunks produced")
        
    target = out / "prepared_text.jsonl"
    with target.open("w", encoding="utf-8") as handle:
        for i, chunk in enumerate(chunks, 1):
            record = {
                "id": i,
                "source": str(src),
                "text": chunk,
                "page": None
            }
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            
    mapping = [{"source": str(src), "output": str(target)}]
    processing_method = f"{method} text chunks of {size} characters"
    return [target], mapping, processing_method

run_A = register_and_run(
    "documents",
    SOURCE_A,
    inspect_documents,
    prepare_documents,
    CONFIG_A
)
print("Cell 8A completed successfully.")


### Cell 9 — REQUIRED: Repeatability and Provenance Checks (Slide 13)
Reruns the registered pipeline to verify deterministic reproducibility of input and output checksums.


In [ ]:
# Cell 9 — REQUIRED: Repeatability and Provenance Verification
assert PIPELINE_RUNS, "No pipeline runs registered. Run at least one selected Cell 8 pathway first."

print("--- Repeatability Verification ---")
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    second = entry["runner"]()
    
    assert first["sources"] == second["sources"], f"Source fingerprint changed for {kind}"
    assert first["validation"] == second["validation"], f"Prepared output changed for {kind}"
    assert first["source_output_mapping"] == second["source_output_mapping"], f"Source-to-output mapping changed for {kind}"
    
    print(f"PASS: {kind} — source fingerprints, prepared outputs and mappings match")

print("\nAll selected pipelines passed repeatability smoke checks.")
